# Experiments — Comparacao de Modelos e Hyperparameter Tuning

Avaliacao sistematica dos tres algoritmos com cross-validation e grid search.

| Secao | Conteudo |
|-------|---------|
| 1 | Cross-validation 5-fold em todos os modelos |
| 2 | GridSearchCV no RandomForest |
| 3 | Curvas ROC comparativas |
| 4 | Matriz de confusao |
| 5 | MLflow tracking dos experimentos |


## Setup


In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import mlflow
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, roc_curve
import matplotlib.pyplot as plt

from src.features.build_features import build_features
from src.models.model_utils import split_features_target

df = build_features(
    input_csv='../data/raw/ecommerce_customer_churn_dataset.csv',
    output_csv='../data/processed/features.csv',
)
X_train, X_test, y_train, y_test = split_features_target(df)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')


## 1 — Cross-Validation 5-fold


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'LogisticRegression':  LogisticRegression(max_iter=1000),
    'RandomForest':        RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting':    GradientBoostingClassifier(n_estimators=100, random_state=42),
}

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:25s}  AUC={scores.mean():.4f} (+/- {scores.std():.4f})')


## 2 — GridSearchCV no RandomForest


In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth':    [None, 10, 20],
    'min_samples_split': [2, 5],
}

gs = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
)
gs.fit(X_train, y_train)

print(f'Melhores parametros: {gs.best_params_}')
print(f'Melhor AUC (CV):     {gs.best_score_:.4f}')
best_rf = gs.best_estimator_


## 3 — Curvas ROC comparativas


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

# Melhor RF (GridSearch)
proba_best = best_rf.predict_proba(X_test)[:, 1]
fpr_b, tpr_b, _ = roc_curve(y_test, proba_best)
auc_b = roc_auc_score(y_test, proba_best)
ax.plot(fpr_b, tpr_b, '--', label=f'RF Tuned (AUC={auc_b:.3f})')

ax.plot([0,1],[0,1],'k:', alpha=0.4)
ax.set_xlabel('FPR')
ax.set_ylabel('TPR')
ax.set_title('Curvas ROC — Comparacao de Modelos')
ax.legend()
plt.tight_layout()
plt.show()


## 4 — Matriz de Confusao (Melhor Modelo)


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

y_pred = best_rf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Nao churn', 'Churn']))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Nao churn', 'Churn'])
plt.title('Matriz de Confusao — RandomForest Tuned')
plt.show()


## 5 — MLflow Tracking dos Experimentos


In [ ]:
mlflow.set_tracking_uri('sqlite:///../notebooks/mlflow.db')
mlflow.set_experiment('experiments_comparacao')

for name, model in {**models, 'RF_Tuned': best_rf}.items():
    if name != 'RF_Tuned':
        model.fit(X_train, y_train)
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    prec = model.score(X_test, y_test)

    with mlflow.start_run(run_name=name):
        mlflow.log_metric('roc_auc', auc)
        mlflow.log_param('model_type', name)
        if hasattr(model, 'n_estimators'):
            mlflow.log_param('n_estimators', model.n_estimators)
        print(f'{name:25s}  AUC={auc:.4f}')

print('\nExperimentos registrados no MLflow.')
